# Kronig-Penny model

 Update history
 - 2021.01.01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
 - 2024.08.07 : Minsu Jeong,  KAIST Electrical Engineering, Write Kronig-Penny code based on TISE. 
 - 2025.03.27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization (plotly) and the description. 
 - 2025.09.30 : Minsu Jeong,  KAIST Electrical Engineering, added effective mass
 - 2026.09.03 : Seungho Chung, updated the entire codebse for python 3.14 version
 - 2026.09.17 : Minsu Jeong,  KAIST Electrical Engineering, Added sparse Bloch bands calculation (scipy) and defect cases.

 ref
1. R. de L. Kronig and W. G. Penney, *Quantum mechanics of electrons in crystal lattices*, Proceedings of the Royal Society A **130**, 499–513 (1931). [DOI](https://doi.org/10.1098/rspa.1931.0019).
2. Stefan Birner, nextnano, [Dispersion in infinite superlattices: Minibands (Kronig–Penney model)](https://nextnano.de/nextnano3/tutorial/1Dtutorial14.htm).
3. Bengt Fornberg, *Generation of finite difference formulas on arbitrarily spaced grids*, Mathematics of Computation **51**, 699–706 (1988). [DOI](https://doi.org/10.1090/S0025-5718-1988-0935077-0). Background on finite-difference weights; this notebook retains its original coefficient generator.
4. SciPy, [`scipy.sparse.linalg.eigsh`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html).
5. SciPy, [`scipy.sparse.bmat`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.bmat.html).
6. MIT 6.763, [Lecture 9: Physical meaning of the wavefunction](https://www.ocw.mit.edu/courses/6-763-applied-superconductivity-fall-2005/9a9a90e0f02022bd3c47da9e6dc1a628_lecture9.pdf), p. 5.
7. C. Kittel, *Introduction to Solid State Physics*, 8th ed., John Wiley & Sons, p. 169.
8. D. A. Neamen, *Semiconductor Physics and Devices: Basic Principles*, 4th ed., McGraw-Hill (2012), p. 63.
9. John R. Hiller, *Quantum Mechanics Simulations*, The Consortium for Upper-Level Physics Software, John Wiley & Sons.

---


## Usage Guide

### Choose a calculation mode

1. Execute the code cell below to display the controls.
2. Choose a mode and set the reference potential, effective mass, and number of displayed bands. The reference potential is the repeating potential before any local wells are replaced.
3. Click **Run calculation** to apply the current settings. Changing a control alone does not redraw the results.

| Mode | Toggle settings | Calculation and output |
| --- | --- | --- |
| Finite wells | **PBC** off | 1–15 wells in a closed box; eigenstates and discrete energy levels |
| Periodic reference potential | **PBC** on, **Defect** off | Six repeating cells; Gamma eigenstates and numerical E–k bands; analytic KP comparison for a square reference potential |
| Local defects | Both toggles on | A square reference potential with one or two replacement wells; Gamma eigenstates, numerical E–k bands, and bands for the reference potential |

The buttons show **PBC (on)** / **PBC (off)** and **Defect (on)** / **Defect (off)**. Turning **PBC** off also turns **Defect** off. The finite-well count is hidden in PBC mode. Defect mode selects and locks the reference potential to **Square Well**.

### Parameters and controls

For a square reference potential, the well bottom is $-D$ and the barrier is zero. Lengths are in angstroms (Å); plotted energies are in eV.

| Symbol | Control or definition | Meaning and unit |
| --- | --- | --- |
| $D$ | **Barrier Height [eV]:** | Positive reference potential well depth, in eV; bottom $-D$, barrier $0$ |
| $a$ | **Well Width [Ang]:** | Square reference potential well width, in Å |
| $b$ | **Barrier Width [Ang]:** | Square reference potential barrier width, in Å |
| $m_{\rm eff}$ | **Effective mass** | Dimensionless mass ratio $m^*/m_e$ |
| $\ell$ | $a+b$ for a square reference potential | Primitive cell period, in Å |
| $L$ | $6\ell$ in PBC mode | Six-cell supercell length, in Å |
| $D_{\rm defect}$ | Defect **Depth [eV]:** | Positive replacement-well depth; bottom $-D_{\rm defect}$ |
| $w_{\rm defect}$ | Defect **Width [Å]:** | Total replacement-well support width, in Å; not the full width at half maximum |
| $\eta$ | **Flat-bottom fraction:** | Flat-bottom width divided by total defect width, from 0 to 1 |

The non-square reference potential options retain their original potential definitions and use the well-width parameter as their cell period, so their $\ell=a$ and $L=6a$. The square reference potential meanings of $a$ and $b$ do not describe every non-square profile.

| Control | How to use it |
| --- | --- |
| **Potential Shape:** | Select the reference potential shape; trapezoidal is a local-defect option only |
| **# of potential well:** | Number of finite wells, from 1 to 15 |
| **# of Bands:** | Number of state groups to display; each periodic group has six branches |
| **Defect count:** | Replace one or two cells |
| **Separation:** | Minimum periodic separation of two defects, in cells |
| **Independent settings for two defects** | Off: both sites use Defect 1 settings; on: configure Defect 2 separately |
| Defect **Shape:** | Square, Symmetric triangular, Parabolic, or Trapezoidal |
| **Grid points/cell:** | Spatial grid size: 251, 501, 1001, or 2001; ordinary default 251, defect default 501 |
| **Bloch points:** | Number of sampled momenta: 21, 41, 81, or 161; default 21; used in PBC mode |

The defect width must be positive and no greater than the reference potential cell period. The controls place the replacement wells near the supercell center.

### Display a trapezoidal defect

1. Enable **PBC**, then **Defect**.
2. Under **Defect 1**, choose **Shape → Trapezoidal** and **Flat-bottom fraction → 0.5**.
3. Click **Run calculation**. Inspect the black **Potential** curve near the center; the gray dashed **Reference potential** is the unchanged square reference potential.

A fraction of 0 gives a triangle, 1 gives a square, and a value between them gives a trapezoid. Zoom into the well or hide eigenstate curves through the legend to inspect its shape.

## Basic Theory

### An electron in a one-dimensional potential

The stationary Schrödinger equation determines an energy $E$ and wavefunction $\psi(x)$ for a potential energy $V(x)$:

$$H\psi=E\psi,\qquad H=-\frac{\hbar^2}{2m^*}\frac{d^2}{dx^2}+V(x),\qquad m^*=m_{\rm eff}m_e.$$

Here $H$ is the Hamiltonian (kinetic plus potential energy), $m_e$ is the free-electron mass, $m^*$ is the effective mass, and $\hbar$ is the reduced Planck constant. An eigenstate is a wavefunction satisfying this equation at a particular energy. The probability density is proportional to $|\psi(x)|^2$, whereas the plotted wavefunction has a sign. [6–9]

### Wells, barriers, and the energy reference

For the square reference potential, a cell contains a well of width $a$ and a barrier of width $b$:

```text
Potential V(x)

 0 ────┐       ┌─────┐       ┌─────┐       ┌─────► x
       │       │     │       │     │       │
-D     └───────┘     └───────┘     └───────┘
       |   a   |  b  |   a   |  b  |   a   |
       <─────────────>
          period ℓ
```

The control named **Barrier Height [eV]** sets the positive well-to-barrier difference $D$. The numerical plots place the barrier at zero and the well bottom at $-D$. A convention with well bottom zero and barrier $D$ instead uses $E_{\rm formula}=E_{\rm numerical}+D$. Adding this constant changes the energy reference and preserves the wavefunctions.

### From finite levels to bands

A finite set of wells has discrete states in a closed box. This notebook adds four cells on each side of the selected wells before imposing the closed boundaries.

A repeating potential admits Bloch states labeled by a wavevector $k$. Their energies $E_n(k)$ form bands; $n$ is the energy-ranked state index at each $k$. A band gap is an energy interval between the highest energy of a lower group and the lowest energy of the next group, over all sampled momenta. Gamma states show only $k=0$ and need not occur at the gap edges. [1, 2, 7, 8]

For the square reference potential, the primitive period is $\ell=a+b$. The PBC calculation uses a supercell containing six primitive cells, of length $L=6\ell$. Representing the same reference potential in this larger cell folds each primitive band into six branches within the smaller supercell Brillouin zone. These branches are part of the supercell representation.

### What a local defect changes

Defect mode replaces the well in one or two selected cells. It keeps the other cells, the cell length, and effective mass fixed. Because the supercell repeats, the model describes a periodically repeated defect arrangement, even though the original primitive-cell periodicity is broken.

The replacement depth, width, and shape modify the potential and may shift or introduce states. Equal depth and width for different shapes do not imply equal integrated potential. Depth changes alone do not specify donor/acceptor species or determine n-type/p-type carrier doping.

The defect gap uses the first six occupied branches as its fixed reference and measures the separation between E6 and E7 over momentum. With implicit spin degeneracy, that reference corresponds to 12 electrons per supercell. Defect-induced states participate in this gap definition; it is not obtained by removing those states from the spectrum.


<details>
<summary><strong>Advanced Theory</strong></summary>

### Spatial grid and the second-derivative matrix

Let $N$ be the number of spatial grid points, $dx$ the spacing between neighboring points, $x_i$ the position of grid point $i$, and $\psi_i=\psi(x_i)$ the sampled wavefunction. The column vector $\boldsymbol\psi=(\psi_0,\ldots,\psi_{N-1})^T$ contains these samples.

**$D_2$ denotes an $N\times N$ matrix approximating the second spatial derivative, not the well depth $D$.** The subscript 2 specifies the derivative order:

$$
[D_2\boldsymbol\psi]_i\approx\psi''(x_i).
$$

For illustration, the three-point centered expression is

$$
\psi''(x_i)\approx\frac{\psi_{i-1}-2\psi_i+\psi_{i+1}}{dx^2}.
$$

An interior row of its matrix therefore has the entries $1,-2,1$, divided by $dx^2$, in columns $i-1,i,i+1$. **The implemented stencil uses seven points**, with offsets $s=-3,\ldots,3$ and dimensionless weights $c_s$:

$$
\psi''(x_i)\approx\frac{1}{dx^2}\left[-\frac{49}{18}\psi_i+\frac32(\psi_{i-1}+\psi_{i+1})-\frac3{20}(\psi_{i-2}+\psi_{i+2})+\frac1{90}(\psi_{i-3}+\psi_{i+3})\right].
$$

Thus $c_0=-49/18$, $c_{\pm1}=3/2$, $c_{\pm2}=-3/20$, and $c_{\pm3}=1/90$. The existing coefficient generator supplies these weights. Boundary rows must implement the chosen closed or Bloch boundary condition. [3]

The defect path uses the midpoint grid $N=6\times\text{points per cell}$, $dx=L/N$, and $x_i=-L/2+(i+1/2)dx$, with $i=0,\ldots,N-1$ and no duplicate endpoint.

### Bloch boundary conditions and boundary matrix entries

For a potential repeated with supercell period $L$, $u_{nk}(x)$ is the periodic part of a Bloch wavefunction, and $\phi=kL$ is the phase accumulated across the supercell:

$$
\psi_{nk}(x)=e^{ikx}u_{nk}(x),\qquad u_{nk}(x+L)=u_{nk}(x),
$$
$$
\psi_{nk}(x+L)=e^{i\phi}\psi_{nk}(x),\qquad \psi'_{nk}(x+L)=e^{i\phi}\psi'_{nk}(x).
$$

Ordinary periodic boundary conditions are the Gamma case $\phi=0$. For a real scalar potential, $E_n(-k)=E_n(k)$, so this notebook samples $0\leq\phi\leq\pi$, displayed as $kL/\pi$. Each momentum has its own Hamiltonian solution. [1, 2]

For row $i$ and stencil offset $s$, let $j$ be the wrapped column index and $\nu$ the signed number of supercell crossings:

$$
i+s=j+\nu N,\qquad j=(i+s)\bmod N,\qquad \nu=\left\lfloor\frac{i+s}{N}\right\rfloor.
$$

The Bloch relation replaces an out-of-cell sample by its wrapped value:

$$
\psi_{i+s}=e^{i\nu\phi}\psi_j,\qquad [D_2(\phi)]_{ij}\mathrel{+}=\frac{c_s}{dx^2}e^{i\nu\phi}.
$$

The notation $\mathrel{+}=$ means adding this stencil contribution to a matrix entry. A right crossing carries $e^{+i\phi}$ and a left crossing $e^{-i\phi}$. Second- and third-neighbor links crossing the boundary also carry their phases. Opposite links are complex conjugates, so the Hamiltonian is Hermitian. In the closed-boundary calculation, stencil samples outside the finite box are zero.

### Sparse Hamiltonian and shift–invert eigenvalues

The diagonal matrix $\operatorname{diag}(V)$ places $V(x_i)$ on its diagonal. In the defect path, lengths are in Å and energies in eV. Define $C=\hbar^2/(2m_e)\simeq3.809982\,\mathrm{eV}\,\mathrm{Å}^2$ in these units. Then

$$
H(\phi)=-\frac{C}{m_{\rm eff}}D_2(\phi)+\operatorname{diag}(V).
$$

Since $D_2$ has units Å⁻², the kinetic term has units eV. The ordinary path instead uses effective atomic units internally, where the kinetic matrix is $-D_2/2$; its plotted energies are converted to eV.

Each row has seven stencil links. Sparse storage therefore scales as $O(N)$, rather than storing an entire $N\times N$ array. The number of low states requested is set by the displayed groups and gap calculation, rather than computing the whole spectrum. [3, 4]

Let $I$ be the identity matrix and $\sigma$ an energy shift below the potential minimum. Shift–invert replaces the eigenvalues by

$$
(H-\sigma I)^{-1}\boldsymbol\psi_n=\frac{1}{E_n-\sigma}\boldsymbol\psi_n.
$$

Because $\sigma$ lies below the spectrum, the largest transformed magnitudes correspond to the lowest original energies. In the defect path, $\sigma=\min(V)-1\,\mathrm{eV}$. This is a solver shift; returned energies keep the original potential reference. The solver applies the inverse through sparse factorization and solves, rather than forming a dense inverse matrix. [4]

### Real representation of a complex Bloch Hamiltonian

For general phases, let $A=\operatorname{Re}H$, $B=\operatorname{Im}H$, $u=\operatorname{Re}\boldsymbol\psi$, and $v=\operatorname{Im}\boldsymbol\psi$. Then

$$
H\boldsymbol\psi=E\boldsymbol\psi\quad\Longleftrightarrow\quad\begin{pmatrix}A&-B\\B&A\end{pmatrix}\begin{pmatrix}u\\v\end{pmatrix}=E\begin{pmatrix}u\\v\end{pmatrix}.
$$

Hermiticity implies $A^T=A$ and $B^T=-B$. The resulting $2N\times2N$ matrix is real symmetric and remains sparse. Each physical eigenvalue appears twice because $\boldsymbol\psi$ and $i\boldsymbol\psi$ give independent real vectors. Combining sorted representation pairs retains physical degeneracies; it does not discard all equal energies. At $\phi=0,\pi$, the original real $N\times N$ matrix is used directly. Gamma wavefunctions therefore come from the original-dimensional vectors. [4, 5]

### Analytic Kronig–Penney bands and supercell folding

For the square reference potential, let $K$ be the primitive-cell Bloch wavevector, $p$ the wavevector inside the well, and $q$ the decay parameter inside the barrier below zero energy. With $E$ measured relative to the zero-energy barrier, continuity of the wavefunction and derivative gives

$$
F(E)=\cos(pa)\cosh(qb)+\frac{q^2-p^2}{2pq}\sin(pa)\sinh(qb)=\cos(K\ell),
$$
$$
p=\frac{\sqrt{2m^*(E+D)}}{\hbar},\qquad q=\frac{\sqrt{-2m^*E}}{\hbar}.
$$

The expression extends above the barrier through complex arithmetic and uses limiting forms when $p$ or $q$ vanishes. Allowed intervals satisfy $|F(E)|\leq1$; their edges satisfy $F(E)=\pm1$. Analytic E–k energies are roots of this relation. The numerical energies come independently from the discretized Hamiltonian. [1, 2]

Here $k$ denotes the supercell wavevector, unlike the primitive wavevector $K$. With the integer folding index $r=0,\ldots,5$,

$$
K=k+\frac{2\pi r}{L},\qquad L=6\ell.
$$

These primitive momenta give the same supercell Bloch phase. Each primitive band yields six folded branches. Both E–k panels use the same supercell momentum axis.

Let $j$ be a one-based displayed-band-group index and $E_n(k)$ the energy of state $n$ in ascending order at that momentum. The numerical adjacent-group gap is

$$
E_{g,j}=\max\left(0,\min_kE_{6j+1}(k)-\max_kE_{6j}(k)\right).
$$

The code ranks states independently at each momentum; it does not track an eigenvector continuously through band crossings.

### Defect profiles, filling, and site density

For a replacement well of depth $D_{\rm defect}$ and total width $w_{\rm defect}$, the magnitudes of the integrated potentials are $D_{\rm defect}w_{\rm defect}$ (square), $D_{\rm defect}w_{\rm defect}/2$ (triangle), $2D_{\rm defect}w_{\rm defect}/3$ (parabola), and $D_{\rm defect}w_{\rm defect}(1+\eta)/2$ (trapezoid). The dimensionless parameter $\eta$ is the flat-bottom fraction.

Let $E_v$ and $E_c$ denote the upper and lower edges at the fixed six-branch filling reference. The gap and its change relative to the reference potential are

$$
E_v=\max_kE_6(k),\qquad E_c=\min_kE_7(k),
$$
$$
E_g=\max(0,E_c-E_v),\qquad\Delta E_g=E_g^{\rm defect}-E_g^{\rm normal}.
$$

Reference and defect calculations use the same reference potential parameters, grid, momentum sampling, and effective mass. This reference corresponds to 12 electrons per supercell with implicit spin degeneracy. Defect states are included in the energy ranks.

For $N_{\rm defect}$ configured replacement sites,

$$
n_{\rm defect}=\frac{N_{\rm defect}}{L},\qquad1\,\mathrm{Å}^{-1}=10^8\,\mathrm{cm}^{-1}.
$$

A site matching the reference potential still counts as a configured replacement. Changing separation at fixed site count and length preserves this site density.

### Wavefunction normalization and display phase

The sampled eigenvectors satisfy $\sum_i|\psi_i|^2=1$. A continuous probability density is represented by $|\psi_i|^2/dx$. The plotted signed amplitude $E_j+10\psi_j(x)$ is a visualization, rather than a probability-density curve. [6]

Before drawing, the largest absolute component is chosen positive. Multiplication by a constant phase preserves energy and probability density; this choice changes only the displayed vector. Degenerate states may use different eigenvector bases.

</details>

<details>
<summary><strong>Code Implementation</strong></summary>

### Calculation flow and entry points

The **Run calculation** button calls `run_kp_calculation`. The callback reads the widgets and passes explicit settings to `main` for finite or periodic potentials, or to `main_defect` for local defects. Both calculation entry points are grouped before the simulation controls. Both routes assemble sparse Hamiltonians and use `real_bloch_eigenpairs`.

| Stage | Finite / periodic potential | Local defects |
| --- | --- | --- |
| Settings | `main` reads reference potential values, mode, displayed bands, grid size, and Bloch-point count | `current_defect_settings` supplies explicit reference and replacement-well settings to `main_defect` |
| Grid and potential | `main` sets the box and effective-unit conversions; `potential` supplies the original reference potential profiles | `defect_potential` builds the six-cell midpoint grid; `well_profile` supplies centered replacement profiles |
| Hamiltonian | `laplacianCoeff` → `laplacian` → `hamiltonian` | `laplacianCoeff` → `bloch_hamiltonian` |
| Low eigenvalues | Finite: `solve_tise`; periodic: `solve_periodic_bands` | `solve_defect_bands`, with `normal_defect_reference` for the unchanged reference potential |
| Analytic comparison | Square periodic reference potential: `kp_functional` → `kp_band_edges` → `analytic_kp_bands`, through `solve_and_plot_analytic` | Numerical defect and reference bands |
| Plot and result | `plot_supercell`; result in `legacy_result` | `main_defect` calls `plot_defect_result`; result in `defect_result` |

### Ordinary calculation

`main` sets the total cell count to the selected finite-well count plus eight buffer cells, or to six in PBC mode. It converts input Å and eV values to effective atomic units using the chosen mass, then assembles the spatial grid and potential.

`laplacianCoeff` uses the existing coefficient generator; `laplacianDim = 3` selects the seven-point stencil. `laplacian` assembles row, column, and coefficient arrays as COO entries and converts the matrix to CSC. In finite mode, out-of-box links are omitted. In PBC mode, wrapped links are multiplied by their Bloch phases. `hamiltonian` adds the diagonal potential.

`solve_tise` requests `finite well count × band count` states. `solve_periodic_bands` requests `6 × band count + 1` states at each sampled phase, keeping the Gamma vectors for the eigenstate plot. The additional branch supports gap evaluation.

For a square periodic reference potential, `kp_band_edges` finds analytic allowed intervals and `analytic_kp_bands` solves the dispersion relation with `brentq` for the folded momenta. `solve_and_plot_analytic` also displays the KP functional. `main` calculates adjacent-group numerical gaps and compares them with the analytic band-edge gaps.

`legacy_result` contains `eigenvalues`, `eigenvectors`, `periodic`, `analytic`, `gaps`, `points_per_cell`, `nk`, and `energy_to_ev`. Multiply its effective-unit energies by `energy_to_ev` to obtain eV. In PBC mode, its standalone eigenvalues and eigenvectors are the Gamma states; `periodic` contains the k-resolved energies.

### Defect calculation and reference potential

`current_defect_settings` returns `host = (depth, well width, barrier width, effective mass)` and one or two replacement specifications `(depth, width, shape, flat-bottom fraction)`. With independent settings off, two sites share Defect 1 settings.

`defect_sites` selects cell 2 for one defect, or cells (2,3), (1,3), and (1,4) for separations of 1, 2, and 3 cells. These are zero-based indices. `defect_potential` first builds a square reference potential cell, then replaces the selected cells with centered `well_profile` wells. All other cells are unchanged.

`bloch_hamiltonian` constructs the seven-point kinetic matrix directly in eV and Å and adds the diagonal defect potential. `solve_defect_bands` calls it at each sampled phase, retaining Gamma eigenvectors. It requests `max(7, 6 × band count) + 1` states, while the plot displays the requested groups and includes E7 in E–k for the reference gap.

`normal_defect_reference` caches up to eight unchanged reference potential results, keyed by reference potential parameters, points per cell, Bloch-point count, and band count. `main_defect` validates its explicit inputs, calculates the reference and defect bands, compares their gaps, and adds `normal_gap`, `delta_gap`, `defect_density_ang_inv`, and `defect_density_cm_inv` to `defect_result`. It prints the summary, displays the comparison plot, and returns the result. These operations do not read widget values.

`defect_result` also contains positions `x`, `potential`, spacing `dx`, `phases`, k-resolved `energies`, `gamma_vectors`, edges `ev` and `ec`, `gap`, `raw_gap`, and the calculation settings. Energies are already in eV. Its `energies` array has one row per sampled momentum and one column per energy-ranked state.

### Eigensolver, energy sorting, and multiplicities

`real_bloch_eigenpairs` uses the real original matrix at Gamma and the zone boundary. At other phases, `bmat` builds the doubled real symmetric matrix. `eigsh` uses shift–invert with `which='LM'` and a shift below the potential minimum.

Returned energies are sorted in ascending order. When vectors are returned, the same permutation is applied to their columns. For the doubled representation, twice the requested physical count is solved and adjacent representation pairs are combined. Physical degeneracies remain present. No operation reduces the spectrum to unique energies.

Consequently, state columns identify energy rank at each momentum. They do not label a state followed through crossings. **Eig i** is zero-based and corresponds to the one-based energy $E_{i+1}$ used in the theory; **Band j** is one-based.

### Plotting and widget behavior

`wavefunction_for_plot` applies the display-only sign convention, leaving stored vectors unchanged. Both plot functions add $10\psi$ to the actual energy and draw a gray dotted line at that energy.

In PBC and defect modes, the color group is `state index // 6`; finite mode uses `state index // finite well count`. `list_color` supplies red, green, blue, magenta, yellow, and cyan. Defect plots overlay normal-reference bands as faint dotted curves in matching colors.

`energy_plot_range` includes potentials, true energies, displayed wavefunction excursions, and the displayed bands. Plotly energy axes share that range and linked zoom. `halve_bloch_panel_widths` reduces E–k panel widths while retaining the eigenstate panel width. The legend can hide individual eigenstates; defect band legends toggle each band group.

`update_defect_widgets` controls mode-dependent visibility, the square reference potential lock, independent second-defect settings, and the trapezoidal flat-bottom control. `run_kp_calculation` reads the current settings when clicked and calls `main` or `main_defect`. It manages the output area and Run button state, and displays exceptions raised by either calculation route.

</details>


In [ ]:
'''
 Update history
 - 2021.01.01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
 - 2024.08.07 : Minsu Jeong,  KAIST Electrical Engineering, Write Kronig-Penny code based on TISE. 
 - 2025.03.27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization (plotly) and the description. 
 - 2025.09.30 : Minsu Jeong,  KAIST Electrical Engineering, added effective mass
 - 2026.09.03 : Seungho Chung, updated the entire codebse for python 3.14 version
 - 2026.09.17 : Added sparse Bloch bands and local-defect calculations; unified plots and English documentation.

 ref
1. R. de L. Kronig and W. G. Penney, *Quantum mechanics of electrons in crystal lattices*, Proceedings of the Royal Society A **130**, 499–513 (1931). [DOI](https://doi.org/10.1098/rspa.1931.0019).
2. Stefan Birner, nextnano, [Dispersion in infinite superlattices: Minibands (Kronig–Penney model)](https://nextnano.de/nextnano3/tutorial/1Dtutorial14.htm).
3. Bengt Fornberg, *Generation of finite difference formulas on arbitrarily spaced grids*, Mathematics of Computation **51**, 699–706 (1988). [DOI](https://doi.org/10.1090/S0025-5718-1988-0935077-0). Background on finite-difference weights; this notebook retains its original coefficient generator.
4. SciPy, [`scipy.sparse.linalg.eigsh`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html).
5. SciPy, [`scipy.sparse.bmat`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.bmat.html).
6. MIT 6.763, [Lecture 9: Physical meaning of the wavefunction](https://www.ocw.mit.edu/courses/6-763-applied-superconductivity-fall-2005/9a9a90e0f02022bd3c47da9e6dc1a628_lecture9.pdf), p. 5.
7. C. Kittel, *Introduction to Solid State Physics*, 8th ed., John Wiley & Sons, p. 169.
8. D. A. Neamen, *Semiconductor Physics and Devices: Basic Principles*, 4th ed., McGraw-Hill (2012), p. 63.
9. John R. Hiller, *Quantum Mechanics Simulations*, The Consortium for Upper-Level Physics Software, John Wiley & Sons.
'''

###############################################################################
# Imports
###############################################################################

import numpy as np
import numpy.linalg as lin
import math
import ipywidgets as widgets
from functools import lru_cache
from ipywidgets import interact_manual, IntSlider, FloatSlider, Dropdown, Layout
from scipy.constants import physical_constants, m_e, electron_volt
from scipy.signal import argrelextrema
from scipy.sparse import coo_matrix, diags, bmat
from scipy.sparse.linalg import eigsh
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

np.set_printoptions(threshold=784, linewidth=np.inf)


###############################################################################
# Global Constants and Unit Conversions
###############################################################################

hbar = physical_constants["reduced Planck constant"][0]

bohr_radius     = physical_constants['Bohr radius'][0]  # in miter
angstrom_radius = 1e-10  # in miter
eff_ang2bohr    = 1e-10 / bohr_radius  # ~1.88973

hartree_energy = physical_constants['Hartree energy'][0]  # in Jule
ev_energy      = physical_constants['electron volt'][0]  # in Jule
eff_har2ev     = hartree_energy / ev_energy  # ~27.21140795

defect_result = None

# Global finite-difference stencil setting
laplacianDim = 3
list_color   = ['red', 'green', 'blue', 'magenta', 'yellow', 'cyan']

num_well_lr     = 4  # left and right region to avoid the effect of the box
num_well_kriong = 15
'''
PBC mode explicitly selects the six-cell Bloch calculation.
Otherwise, num_well specifies finite wells inside an infinite potential box.
'''

ngirde          = 2500  # resolution of energy for the analytic functional plot
ngridx_per_cell = 251  # number of grid points per cell

DEFECT_CELLS           = 6
DEFECT_POINTS_PER_CELL = 501
DEFECT_SHAPES          = ('Square', 'Symmetric triangular', 'Parabolic', 'Trapezoidal')
KINETIC_EV_ANG2        = hbar**2 / (2 * m_e * electron_volt * 1e-20)

legacy_result = None


###############################################################################
# Laplacian, Potential and Common Eigensolver
###############################################################################


def laplacianCoeff(laplacianDim):
    """
    Compute the coefficients for the Laplacian operator using the Finite Difference Method (FDM).

    This function constructs the finite difference coefficients for the second derivative 
    using central difference formulas. The method is based on solving a linear system 
    that relates the coefficients to the derivatives of a function at grid points.
    """
    size = 2 * laplacianDim + 1
    a    = np.zeros((size, size))
    c    = np.zeros(size)
    c[2] = math.factorial(2)  # For second derivative, 2! = 2
    for i in range(size):
        for j in range(size):
            a[i, j] = (j - laplacianDim) ** i
    inva   = lin.inv(a)
    b      = inva @ c  # b = a^-1 * c
    lcoeff = b[laplacianDim:size]
    return lcoeff


def laplacian(ngridx, laplacianDim, dx, num_well, is_kroing, phase=0.):
    """
    Construct the sparse finite-difference Laplacian on the existing grid.
    Finite wells use closed boundaries; periodic wells use a Bloch phase.
    """
    lcoeff    = laplacianCoeff(laplacianDim)
    rows      = np.repeat(np.arange(ngridx), 2 * laplacianDim + 1)
    offsets   = np.tile(np.arange(-laplacianDim, laplacianDim + 1), ngridx)
    unwrapped = rows + offsets

    # Dirichlet (Closed) Boundary Condition
    if not is_kroing:
        valid = (unwrapped >= 0) & (unwrapped < ngridx)
        return coo_matrix((lcoeff[np.abs(offsets[valid])] / dx**2, (rows[valid], unwrapped[valid])),
                          shape=(ngridx, ngridx)).tocsc()

    # Bloch Boundary Condition: psi(x+L) = exp(i*phase) psi(x)
    # Include every crossing of the seven-point stencil, not only nearest neighbors.
    columns = unwrapped % ngridx
    wraps   = np.floor_divide(unwrapped, ngridx)
    data    = lcoeff[np.abs(offsets)] / dx**2 * np.exp(1j * phase * wraps)
    return coo_matrix((data, (rows, columns)), shape=(ngridx, ngridx)).tocsc()


def potential(ngridx, num_well, pot_shape, pot_height_har=25, width_barrier_bohr=2, width_well_bohr=2,
              is_kroing=False):
    """
    Set up the potential for the Kronig-Penny model.
    
    In each cell (period), the barrier region has width 'width_barrier' and 
    the well region has width 'width_well'. The potential barrier height is pot_height_har, 
    and the well region is 0.
    
    Parameters:
    -----------
    ngridx      : int
        Number of grid points.
    num_well    : int
        Number of cells (periods).
    pot_height_har      : float
        Barrier height in Hartree.
    width_barrier_bohr  : float
        Barrier width.
    width_well_bohr     : float
        Well width.
    
    Returns:
    --------
    pot_grid : ndarray
        The potential at each grid point (in Hartree).
    """

    ngridx_per_cell = ngridx // num_well
    pot_grid        = np.zeros(ngridx)

    if pot_shape == 4:
        return pot_grid

    frac_well    = width_well_bohr / (width_barrier_bohr + width_well_bohr)
    frac_barrier = 1 - frac_well

    i_elec_lr = num_well_lr if not is_kroing else 0

    ### Set up offset (for center aligned potential) & length of well
    '''
            barrier  well
           <-------><---->
           ┌-------┐     ┌----- height
           │       │     │
           │       │     │
        ---┘       └-----┘      0
        -------┼---------------->
             center
            <-->
           offset
               
    '''
    # Square well & Triangualr well
    if pot_shape == 0:
        offset   = 0.5 * frac_barrier * ngridx_per_cell if not is_kroing else 0
        len_well = frac_well * ngridx_per_cell

    # Parabolic well (Y-shaped (convex) & U-shaped (concave))
    elif pot_shape == 1 or pot_shape == 2:
        offset   = 0
        len_well = ngridx_per_cell

    # Coulombic
    elif pot_shape == 3:
        if not is_kroing:
            offset   = -(num_well_lr - 0.5) * ngridx_per_cell
            len_well = 2 * num_well_lr * ngridx_per_cell
        else:
            offset   = 0
            len_well = ngridx_per_cell

    ### Set up potential
    for i in range(i_elec_lr, num_well - i_elec_lr):
        start = int(i * ngridx_per_cell + offset)
        end   = int(i * ngridx_per_cell + offset + len_well)

        # Square well
        if pot_shape == 0:
            pot_grid[start:end] -= pot_height_har

        # Triangualr well
        elif pot_shape == 1:
            for ii in range(start, end - 1):
                pot_grid[ii] -= ((ii - start) / (end + 1 - start)) * pot_height_har

        # Parabolic (concave) (U-shaped well)
        elif pot_shape == 2:
            for ii in range(start, end):
                pot_grid[ii] -= (
                    1 - ((ii - (start + end) / 2) / ((end - start) / 2)) ** 2
                ) * pot_height_har

            # Coulomb (legacy)
            # # Parabolic (convex) (Approximation of Coulomic well, Y-shaped well, 1-r**2)
            # elif pot_shape == 3:
            #     for ii in range (start, (end + start)//2):
            #         pot_grid[ii] = ( 1 - ((ii - start) / ((end - start)/2 ))**2 ) * pot_height_har
            #     for ii in range ((end + start)//2, end):
            #         pot_grid[ii] = ( 1 - ((ii - end) / ((end - start)/2 ))**2 ) * pot_height_har

            # Coulomb (soft-core)
            '''
            Used soft core to avoid -inf.

            V(r) = - Z / (r^a + k^a)^(1/a)

            r : radial position
            k : cutoff parameter
            a : order  parameter

            ref - Phys. Rev. A, 80, 032507 (2009)
            '''
        elif pot_shape == 3:
            alpha     = 8  # order of Coulomb potential
            soft_core = 1  # level of soft core (cutoff) of Coulomb potential

            for ii in range(start, end):
                coulombic_dacay = 1 / np.power(np.abs(soft_core**alpha + (ii - (end + start) // 2) ** alpha),
                    1 / alpha)
                pot_coulombic_decay = coulombic_dacay * pot_height_har

                pot_grid[ii] -= pot_coulombic_decay

        # Custom potential   Design your potential :)
        elif pot_shape == 4:
            for ii in range(start, end):
                pot_grid[ii] -= 0

    return pot_grid


def hamiltonian(ngridx, laplacianDim, dx, num_well, pot_shape, pot_height_har=25, width_barrier_bohr=2,
                width_well_bohr=2, is_kroing=False, phase=0.):
    """Construct sparse H = -1/2 * Laplacian + Potential in effective atomic units."""
    pot = potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr,
                    is_kroing)
    laplacian_op = laplacian(ngridx, laplacianDim, dx, num_well, is_kroing, phase)

    ### Effective mass = 1, hbar = 1 (effective atomic units)
    return -laplacian_op / 2. + diags(pot, format='csc')


# Real symmetric representation: H=A+iB -> [[A,-B],[B,A]].
# Each energy appears twice; verified pairs preserve physical degeneracies.
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html
def real_bloch_eigenpairs(ham, nstates, sigma, phase, return_vectors=False):
    real_boundary = abs(phase) < 1e-14 or abs(phase - np.pi) < 1e-14
    if return_vectors and not real_boundary:
        raise ValueError('Physical eigenvectors are returned only at Gamma/pi.')
    if real_boundary:
        matrix = ham.real.tocsc()
        count  = nstates
    else:
        matrix = bmat([[ham.real, -ham.imag], [ham.imag, ham.real]], format='csc')
        matrix.eliminate_zeros()
        count = 2 * nstates
    v0 = np.random.default_rng(20260917).normal(size=matrix.shape[0])
    answer = eigsh(matrix, k=count, sigma=sigma, which='LM', tol=1e-10,
                   ncv=min(matrix.shape[0] - 1, max(4 * count + 1, 40)), v0=v0,
                   return_eigenvectors=return_vectors)
    if return_vectors:
        energies, vectors = answer
        order             = np.argsort(energies)
        return energies[order], vectors[:, order]
    energies = np.sort(answer)
    if not real_boundary:
        pairs = energies.reshape(-1, 2)
        if not np.allclose(pairs[:, 0], pairs[:, 1], rtol=1e-8, atol=1e-7):
            raise RuntimeError('Doubled Bloch eigenvalue pairs did not converge; no result published.')
        energies = pairs.mean(axis=1)
    return energies, None


def solve_tise(ngridx, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, dx, num_well,
               is_kroing, nstates=None):
    """Return only the requested lowest energies and wavefunctions."""
    if nstates is None:
        nstates = num_well if is_kroing else num_well - 2 * num_well_lr
    if not 0 < nstates < ngridx - 1:
        raise ValueError('Requested states must be positive and below the grid size.')
    ham_op = hamiltonian(ngridx, laplacianDim, dx, num_well, pot_shape, pot_height_har, width_barrier_bohr,
                         width_well_bohr, is_kroing)
    # Shift-invert selects the lowest states; the returned energies are unshifted.
    return real_bloch_eigenpairs(
        ham_op,
        nstates,
        float(
            potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr,
                      is_kroing).min()
        )
        - 1.,
        0.,
        return_vectors=True,
    )


###############################################################################
# Local Defect Potential and Bloch Hamiltonian
###############################################################################


def well_profile(relative_x, width, shape='Square', flat_fraction=0.5):
    """Unit-depth centered well; width is the total support, not FWHM."""
    if not np.isfinite(width) or width <= 0:
        raise ValueError('Well width must be positive.')
    if shape not in DEFECT_SHAPES:
        raise ValueError('Unknown defect shape.')
    if not np.isfinite(flat_fraction) or not 0 <= flat_fraction <= 1:
        raise ValueError('Flat-bottom fraction must be between 0 and 1.')
    t      = 2 * np.abs(np.asarray(relative_x, dtype=float)) / width
    inside = t < 1
    if shape == 'Square' or (shape == 'Trapezoidal' and flat_fraction == 1):
        return inside.astype(float)
    if shape == 'Symmetric triangular':
        return np.where(inside, 1 - t, 0)
    if shape == 'Parabolic':
        return np.where(inside, 1 - t**2, 0)
    return np.where(inside, np.minimum(1, (1 - t) / (1 - flat_fraction)), 0)


# Validate reference and defect settings for the selected spatial grid.
def validate_defect_inputs(host, specs, separation, points_per_cell):
    depth, width, barrier, mass = host
    if not np.all(np.isfinite(host)):
        raise ValueError('Reference potential parameters must be finite.')
    if depth < 0 or width <= 0 or barrier < 0 or mass <= 0:
        raise ValueError('Reference potential depth/barrier must be nonnegative; width/mass positive.')
    if len(specs) not in (1, 2) or separation not in (1, 2, 3):
        raise ValueError('Choose one/two defects and a separation of 1, 2, or 3 cells.')
    if int(points_per_cell) != points_per_cell or points_per_cell < 15:
        raise ValueError('Use at least 15 grid points per cell.')
    period = width + barrier
    dx     = period / points_per_cell
    for spec in specs:
        d, w, shape, flat = spec
        if not np.all(np.isfinite([d, w, flat])) or d < 0 or not 0 < w <= period:
            raise ValueError('Defect depth must be nonnegative and 0 < width <= cell period.')
        if w < 4 * dx:
            raise ValueError('Defect width is unresolved: use width >= 4 grid spacings or a finer grid.')
        well_profile(np.array([0.]), w, shape, flat)


def defect_sites(count, separation=1):
    """Place defects near the supercell center, retaining their PBC separation."""
    if count == 0:
        return ()
    if count == 1:
        return ((DEFECT_CELLS - 1) // 2,)
    # A whole-cell translation of (0, separation) preserves the Bloch spectrum.
    # For two cells, put their midpoint as close as possible to x=0.
    first = (DEFECT_CELLS - 1 - separation) // 2
    return (first, first + separation)


def defect_potential(host, specs=(), separation=1, points_per_cell=DEFECT_POINTS_PER_CELL):
    """Replace entire selected cells; unselected reference potential cells stay unchanged."""
    if not specs:
        # Validate the reference potential through an identical square defect.
        validation_specs = ((host[0], host[1], 'Square', 0.5),)
    else:
        validation_specs = specs
    validate_defect_inputs(host, validation_specs, separation, points_per_cell)
    host_depth, host_width, barrier_width, _ = host
    period    = host_width + barrier_width
    dx        = period / points_per_cell
    local_x   = (np.arange(points_per_cell) + 0.5) * dx - period / 2
    host_cell = -host_depth * well_profile(local_x, host_width)
    cells     = np.tile(host_cell, (DEFECT_CELLS, 1))
    sites     = defect_sites(len(specs), separation)
    for site, (depth, width, shape, flat) in zip(sites, specs):
        cells[site] = -depth * well_profile(local_x, width, shape, flat)
    n = DEFECT_CELLS * points_per_cell
    x = (np.arange(n) + 0.5) * dx - DEFECT_CELLS * period / 2
    return x, cells.ravel(), dx


###############################################################################
# THEORY: Bloch finite differences in the *six-cell* supercell, L = 6(a+b).
# psi(x) = exp(ikx) u(x), u(x+L) = u(x) => psi(x+L) = exp(i*phi) psi(x).
# For stencil index i+s = j + wraps*N, psi[i+s] = exp(i*wraps*phi)*psi[j].
# Right crossings therefore get exp(+i*phi), left crossings exp(-i*phi).
# ALL crossings (including second/third neighbors) must carry this phase.
# Opposite links are conjugates, so H(phi) is Hermitian and H(-phi)=H(phi)*.
# phi=0 is ordinary PBC; sweeping phi gives bands, rather than a cosine fit.
#
# H = -(hbar^2 / 2m*) D2 + diag(V), with x in angstrom and E in eV.
# Seven-point D2 coefficients: c0=-49/18, c1=3/2, c2=-3/20, c3=1/90.
# Each row has only seven links: sparse O(N) storage, instead of dense O(N^2).
#
# References (full derivations are in the Markdown cell):
# [1] Kronig & Penney, Proc. R. Soc. A 130, 499-513 (1931):
#     https://doi.org/10.1098/rspa.1931.0019
# [2] nextnano, superlattice Bloch boundary conditions and k-resolved solution:
#     https://nextnano.de/nextnano3/tutorial/1Dtutorial14.htm
# [3] Fornberg, Math. Comp. 51, 699-706 (1988), finite-difference background:
#     https://doi.org/10.1090/S0025-5718-1988-0935077-0
###############################################################################
def bloch_hamiltonian(potential_ev, dx_ang, mass, phase):
    """psi(x+L) = exp(i*phase) psi(x), including every stencil boundary crossing."""
    n         = len(potential_ev)
    rows      = np.repeat(np.arange(n), 2 * laplacianDim + 1)
    offsets   = np.tile(np.arange(-laplacianDim, laplacianDim + 1), n)
    unwrapped = rows + offsets
    columns   = unwrapped % n
    wraps     = np.floor_divide(unwrapped, n)
    coeff     = laplacianCoeff(laplacianDim)
    data      = -KINETIC_EV_ANG2 / mass * coeff[np.abs(offsets)] / dx_ang**2
    data      = data * np.exp(1j * phase * wraps)
    kinetic   = coo_matrix((data, (rows, columns)), shape=(n, n)).tocsc()
    return kinetic + diags(potential_ev, format='csc')


###############################################################################
# Bloch Bands and Analytic Kronig-Penney Comparison
###############################################################################


def solve_periodic_bands(ngridx, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, dx, num_well,
                         num_bands, nk=21):
    """Calculate the folded supercell branches at each Bloch wavevector."""
    nstates = num_well * num_bands + 1
    if nstates >= ngridx - 1:
        raise ValueError('Too many requested bands for this grid.')
    phases        = np.linspace(0, np.pi, int(nk))
    energies      = []
    gamma_vectors = None

    ### Calculation
    for index, phase in enumerate(phases):
        ham = hamiltonian(ngridx, laplacianDim, dx, num_well, pot_shape, pot_height_har, width_barrier_bohr,
                          width_well_bohr, True, phase)
        minimum = potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr,
                            True).min()
        levels, vectors = real_bloch_eigenpairs(ham, nstates, float(minimum) - 1., phase,
            return_vectors=index == 0)
        energies.append(levels)
        if index == 0:
            gamma_vectors = vectors
    return dict(phases=phases, energies=np.asarray(energies), gamma_vectors=gamma_vectors)


def kp_functional(energy, depth, a, b):
    """
    Evaluate the analytic KP discriminant using the numerical energy reference.
    The sine-over-argument form removes singularities at E=-depth and E=0.
    """
    energy = np.asarray(energy, dtype=float)
    k      = np.sqrt(2 * (energy + depth) + 0j)
    q      = np.sqrt(-2 * energy + 0j)
    ka, qb = k * a, q * b
    # sinc(z/pi)=sin(z)/z; sinc(i*z/pi)=sinh(z)/z, including z=0.
    value = np.cos(ka) * np.cosh(qb) + (q**2 - k**2) * a * b / 2 * np.sinc(
        ka / np.pi) * np.sinc(1j * qb / np.pi)
    if not np.all(np.isfinite(value)):
        raise ValueError('KP functional overflow: reduce depth/width or revise the parameter range.')
    return value.real


def kp_band_edges(depth, a, b, num_bands):
    """Bracket band edges using extrema of F(E), then solve F(E)=+/-1."""
    period   = a + b
    span     = depth + 0.5 * (np.pi * (num_bands + 2) / period) ** 2
    previous = None
    # Refinement checks guard against missing narrow allowed intervals.
    for refinement in range(7):
        upper = -depth + span
        grid  = np.linspace(-depth, upper, 512 * 2**refinement + 1)
        step  = max(1e-7, span * 1e-7)

        # Estimate the derivative of the analytic KP functional to locate its extrema.
        def slope(e):
            return float(
                (kp_functional(e + step, depth, a, b) - kp_functional(e - step, depth, a, b)) / (2 * step))

        slopes = (
            kp_functional(grid + step, depth, a, b) - kp_functional(grid - step, depth, a, b)
        ) / (2 * step)
        extrema = []
        for j in np.flatnonzero(slopes[:-1] * slopes[1:] < 0):
            extrema.append(brentq(slope, grid[j], grid[j + 1], xtol=1e-12))
        knots  = np.array([-depth] + extrema + [upper])
        values = kp_functional(knots, depth, a, b)
        edges  = []
        for target in (-1., 1.):
            for j in range(len(knots) - 1):
                f0, f1 = values[j] - target, values[j + 1] - target
                if f0 * f1 < 0:
                    edges.append(
                        brentq(lambda e: float(kp_functional(e, depth, a, b)) - target, knots[j],
                               knots[j + 1], xtol=1e-12)
                    )
            # At a closed gap the edge is a double root and must appear twice.
            for j in np.flatnonzero(np.abs(values - target) < 1e-10):
                adjacent_crossing = (
                    j > 0 and (values[j - 1] - target) * (values[j] - target) < 0
                ) or (j < len(knots) - 1 and (values[j + 1] - target) * (values[j] - target) < 0)
                if not adjacent_crossing:
                    multiplicity = 1 if j in (0, len(knots) - 1) else 2
                    edges.extend([knots[j]] * multiplicity)
        edges = np.sort(edges)
        if len(edges) >= 2 * num_bands:
            selected = edges[: 2 * num_bands]
            centers  = (selected[::2] + selected[1::2]) / 2
            if not np.all(np.abs(kp_functional(centers, depth, a, b)) <= 1 + 1e-8):
                previous = None
                continue
            if previous is not None and np.allclose(selected, previous, rtol=1e-9, atol=1e-10):
                return selected
            previous = selected
        else:
            span *= 2
            previous = None
    raise RuntimeError('KP band-edge brackets did not stabilize; analytic comparison unavailable.')


def analytic_kp_bands(depth, a, b, num_bands, phases):
    """Solve the analytic equation at six primitive momenta per supercell k."""
    period = a + b
    # q=k+2*pi*r/L: one primitive band becomes six folded branches.
    angles = (np.asarray(phases)[:, None] + 2 * np.pi * np.arange(6)[None, :]) / 6
    if depth == 0 or b == 0:
        offset  = -depth if b == 0 else 0.
        momenta = angles[:, :, None] + 2 * np.pi * np.arange(-num_bands - 1, num_bands + 2)
        free    = np.sort((0.5 * (momenta / period) ** 2 + offset).reshape(len(phases), -1), axis=1)
        edges = np.array(
            [
                v
                for j in range(num_bands)
                for v in (offset + 0.5 * (j * np.pi / period) ** 2,
                          offset + 0.5 * ((j + 1) * np.pi / period) ** 2)
            ]
        )
        return dict(energies=free[:, : 6 * num_bands], edges=edges)
    edges = kp_band_edges(depth, a, b, num_bands)
    roots = np.empty((len(phases), 6, num_bands))
    for j in range(num_bands):
        lo, hi   = edges[2 * j : 2 * j + 2]
        flo, fhi = float(kp_functional(lo, depth, a, b)), float(kp_functional(hi, depth, a, b))
        for ik, row in enumerate(angles):
            for r, angle in enumerate(row):
                target = float(np.cos(angle))
                if abs(flo - target) < 1e-10:
                    energy = lo
                elif abs(fhi - target) < 1e-10:
                    energy = hi
                else:
                    energy = brentq(lambda e: float(kp_functional(e, depth, a, b)) - target, lo, hi,
                                    xtol=1e-12)
                if abs(float(kp_functional(energy, depth, a, b)) - target) > 1e-7:
                    raise RuntimeError('KP energy root failed the equation residual check.')
                roots[ik, r, j] = energy
    return dict(energies=np.sort(roots.reshape(len(phases), -1), axis=1), edges=edges)


###############################################################################
# Defect Bands and Reference Potential
###############################################################################

###############################################################################
# NUMERICAL EIGENSOLVER AND GAP CONVENTION
# eigsh uses shift-invert: (H-sigma*I)^(-1) psi = 1/(E-sigma) psi.
# sigma=min(V)-1 eV lies below the spectrum (positive-semidefinite kinetic
# operator), so largest transformed magnitudes select the lowest original E.
# sigma is a solver shift, NOT an energy-reference change. CSC permits sparse
# LU. General Bloch phases use the equivalent real symmetric sparse system
# derived in the Markdown cell:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html
# A primitive reference potential band folds into six supercell branches. We keep six occupied
# branches (12 electrons/supercell with implicit spin degeneracy) as a reference:
# raw_gap=min_k(E7)-max_k(E6); gap=max(0, raw_gap). Defect states ARE included.
# This is not a gap of the reference potential alone, an optical gap, or a carrier-doping calculation.
###############################################################################
# H=A+iB, psi=u+iv => [[A,-B],[B,A]] [u,v]^T = E [u,v]^T.
# Hermiticity gives A^T=A and B^T=-B, hence the doubled matrix is symmetric.
# Every physical eigenvalue appears twice (psi and i*psi); sort, verify pairs,
# then keep their means. Taking pairs preserves physical degeneracies.
# Gamma/pi use H.real directly: no doubling, and Gamma vectors remain physical.
# Both representations retain O(N) sparse storage; no dense N-by-N conversion.


def solve_defect_bands(host, specs=(), separation=1, points_per_cell=DEFECT_POINTS_PER_CELL, nk=21,
                       num_bands=2):
    """Return low supercell branches and the neutral first-gap continuation of the reference potential."""
    if int(nk) != nk or nk < 2 or int(num_bands) != num_bands or num_bands < 1:
        raise ValueError('Use integer nk >= 2 and num_bands >= 1.')
    x, potential_ev, dx = defect_potential(host, specs, separation, points_per_cell)
    nstates             = max(DEFECT_CELLS + 1, DEFECT_CELLS * int(num_bands)) + 1
    if nstates >= len(x) - 1:
        raise ValueError('Too many requested bands for this grid.')
    phases        = np.linspace(0, np.pi, int(nk))  # Real scalar potential: E(k) = E(-k).
    levels        = []
    gamma_vectors = None
    for index, phase in enumerate(phases):
        ham = bloch_hamiltonian(potential_ev, dx, host[3], phase)
        # The shift lies below min(V), so the closest states are the lowest states.
        energies, vectors = real_bloch_eigenpairs(ham, nstates, float(potential_ev.min() - 1), phase,
            return_vectors=index == 0)
        levels.append(energies)
        if index == 0:
            gamma_vectors = vectors
    levels  = np.asarray(levels)
    ev      = float(levels[:, DEFECT_CELLS - 1].max())
    ec      = float(levels[:, DEFECT_CELLS].min())
    raw_gap = ec - ev
    gap     = 0. if abs(raw_gap) < 1e-8 else max(0., raw_gap)
    return dict(
        x=x,
        potential=potential_ev,
        dx=dx,
        phases=phases,
        energies=levels,
        gamma_vectors=gamma_vectors,
        ev=ev,
        ec=ec,
        gap=gap,
        raw_gap=raw_gap,
        host=tuple(host),
        specs=tuple(specs),
        separation=separation,
        points_per_cell=points_per_cell,
        nk=nk,
    )


# Cache the unchanged reference-potential bands for defect comparisons.
@lru_cache(maxsize=8)
def normal_defect_reference(host, points_per_cell=DEFECT_POINTS_PER_CELL, nk=21, num_bands=2):
    return solve_defect_bands(host, points_per_cell=points_per_cell, nk=nk, num_bands=num_bands)


###############################################################################
# Plot Functions Using Plotly
###############################################################################


def wavefunction_for_plot(vector):
    """Choose a display sign without changing stored eigenvectors or energies."""
    psi = np.array(vector, dtype=float, copy=True)
    # H psi = E psi also holds for -psi; |psi|^2 is unchanged.
    # Use the largest absolute amplitude as a display-only sign convention.
    # Degenerate states still permit different bases within their eigenspace.
    if psi[np.argmax(np.abs(psi))] < 0:
        psi *= -1
    return psi


def energy_plot_range(*values):
    """One energy range containing potentials, displayed amplitudes and bands."""
    lower = min(float(np.min(value)) for value in values)
    upper = max(float(np.max(value)) for value in values)
    pad   = max(0.1, (upper - lower) * 0.05)
    return [lower - pad, upper + pad]


def halve_bloch_panel_widths(fig, columns):
    """Halve Bloch-panel pixel widths, retaining the left panel and spacing."""
    # Explicit margins keep the domain-to-pixel conversion reproducible.
    fig.update_layout(margin=dict(l=80, r=80))
    inner_width = fig.layout.width - 160
    domains = [
        list(getattr(fig.layout, 'xaxis' + (str(c) if c > 1 else '')).domain)
        for c in range(1, columns + 1)
    ]
    saved_width     = sum((right - left) * inner_width / 2 for left, right in domains[1:])
    new_inner_width = inner_width - saved_width
    removed         = 0.
    for index, (left, right) in enumerate(domains):
        start       = left * inner_width - removed
        width       = (right - left) * inner_width * (0.5 if index else 1.)
        domain      = [start / new_inner_width, (start + width) / new_inner_width]
        axis        = getattr(fig.layout, 'xaxis' + (str(index + 1) if index else ''))
        axis.domain = domain
        fig.layout.annotations[index].x = sum(domain) / 2
        if index:
            fig.layout.annotations[index].text = fig.layout.annotations[index].text.replace(' (', '<br>(')
            removed += (right - left) * inner_width / 2
    fig.update_layout(width=new_inner_width + 160)


def plot_supercell(eigval, eigvec, ngridx, box_bohr, pot_shape, pot_height_har, width_barrier_bohr,
                   width_well_bohr, num_bands, num_well, is_kroing, periodic_result=None,
                   analytic_result=None):
    """Plot energy-anchored wavefunctions, finite levels or separate Bloch bands."""
    # x coordinate in Angstrom
    x = np.linspace(-box_bohr / 2, box_bohr / 2, ngridx) / eff_ang2bohr
    pot = potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr,
                    is_kroing)
    num_well -= 2 * num_well_lr if not is_kroing else 0
    compare_analytic = is_kroing and analytic_result is not None

    # Create subplots
    titles = ["Eigenstates & Potential", "E–k Diagram (Numerical)" if is_kroing else ""]
    if compare_analytic:
        titles.append("E–k Diagram (Analytic)")
    fig = make_subplots(
        rows=1,
        cols=len(titles),
        column_widths=[0.45, 0.275, 0.275] if compare_analytic else [0.75, 0.25],
        subplot_titles=titles,
    )

    ### Left subplot: Eigenstates and potential
    i_eig = []
    for i_band in range(num_bands):
        indices = np.arange(i_band * num_well, (i_band + 1) * num_well)
        # Preserve the existing finite-mode selection of fully bound bands.
        if not is_kroing and eigval[indices].max() > 0:
            continue
        i_eig.extend(indices)

    displayed = [pot * eff_har2ev]
    for i in i_eig:
        energy_ev = eigval[i] * eff_har2ev
        # Display amplitude at the actual E; this scale is not normalization.
        # Subtracting psi[0] would move the reference by a different amount per state.
        psi      = wavefunction_for_plot(eigvec[:, i])
        eigstate = psi * 10 + energy_ev
        displayed.append(eigstate)
        displayed.append(np.array([energy_ev]))
        label = f'Eig {i}'
        fig.add_trace(
            go.Scatter(
                x=x,
                y=eigstate,
                mode='lines',
                line=dict(color=list_color[(i // num_well) % len(list_color)]),
                name=label,
                customdata=np.column_stack((np.full(ngridx, energy_ev), psi)),
                hovertemplate=label
                + f' (E{i+1})'
                + ('; Gamma' if is_kroing else '')
                + '<br>x=%{x:.4f} Å'
                '<br>E=%{customdata[0]:.6f} eV'
                '<br>ψ=%{customdata[1]:.6f}'
                '<br>Display: E + 10×ψ<extra></extra>',
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=[x[0], x[-1]],
                y=[energy_ev, energy_ev],
                mode='lines',
                line=dict(color='gray', dash='dot'),
                name=label + ' energy',
                showlegend=False,
                hovertemplate=label + '<br>E=%{y:.6f} eV<extra></extra>',
            ),
            row=1, col=1,
        )
    fig.add_trace(
        go.Scatter(
            x=x,
            y=pot * eff_har2ev,
            mode='lines',
            line=dict(color='black'),
            name='Potential',
            hovertemplate='x=%{x:.4f} Å<br>V=%{y:.6f} eV<extra></extra>',
        ),
        row=1, col=1,
    )
    fig.update_xaxes(title_text='Position [Angstrom]', row=1, col=1)

    ### Right subplots: Finite levels or numerical and analytic Bloch bands
    if is_kroing:
        phase        = periodic_result['phases'] / np.pi
        numerical_ev = periodic_result['energies'][:, : num_bands * num_well] * eff_har2ev
        analytic_ev  = analytic_result['energies'] * eff_har2ev if compare_analytic else None
        for state in range(num_bands * num_well):
            i_band = state // num_well
            color  = list_color[i_band % len(list_color)]
            # E_j denotes energy rank at each k, not wavefunction tracking at crossings.
            detail = f'Band {i_band+1}; Eig {state} (E{state+1})'
            hover  = detail + '<br>kL/π=%{x:.4f}<br>E=%{y:.6f} eV<extra></extra>'
            fig.add_trace(
                go.Scatter(
                    x=phase,
                    y=numerical_ev[:, state],
                    mode='lines',
                    line=dict(color=color),
                    name=f'Band {i_band+1}',
                    legendgroup=f'numerical_band_{i_band+1}',
                    showlegend=state % num_well == 0,
                    hovertemplate='Numerical H(k)<br>' + hover,
                ),
                row=1, col=2,
            )
            if compare_analytic:
                fig.add_trace(
                    go.Scatter(
                        x=phase,
                        y=analytic_ev[:, state],
                        mode='lines',
                        line=dict(color=color, dash='dot'),
                        name=f'Band {i_band+1} (Analytic)',
                        legendgroup=f'analytic_band_{i_band+1}',
                        showlegend=state % num_well == 0,
                        hovertemplate='Analytic KP roots<br>' + hover,
                    ),
                    row=1, col=3,
                )
        displayed.append(numerical_ev)
        if compare_analytic:
            displayed.append(analytic_ev)
        for column in range(2, len(titles) + 1):
            fig.update_xaxes(title_text='kL / π (six-cell supercell)', range=[0, 1], row=1, col=column)
    else:
        for i_band in range(num_bands):
            indices = np.arange(i_band * num_well, (i_band + 1) * num_well)
            if eigval[indices[-1]] < 0:
                fig.add_trace(
                    go.Scatter(
                        x=indices,
                        y=eigval[indices] * eff_har2ev,
                        mode='markers',
                        marker=dict(size=6, color=list_color[i_band % len(list_color)]),
                        name=f'Band {i_band+1}',
                        hovertemplate='Eig %{x}<br>E=%{y:.6f} eV<extra></extra>',
                    ),
                    row=1, col=2,
                )
        fig.update_xaxes(title_text='Occurance', row=1, col=2)

    # Match all panels, including the potential and arbitrary display amplitudes.
    plot_range = energy_plot_range(*displayed)
    for column in range(1, len(titles) + 1):
        fig.update_yaxes(title_text='Energy [eV]', range=plot_range, matches='y' if column > 1 else None,
                         row=1, col=column)

    fig.update_layout(title_text='Eigenstates, Potential', width=1500 if compare_analytic else 1000,
                      height=600, legend=dict(groupclick='togglegroup'))
    if is_kroing:
        halve_bloch_panel_widths(fig, len(titles))
    fig.show()


def solve_and_plot_analytic(pot_height_har, width_barrier_bohr, width_well_bohr, num_bands, phases):
    """Solve the KP equation and show the original allowed-band functional."""
    result     = analytic_kp_bands(pot_height_har, width_well_bohr, width_barrier_bohr, num_bands, phases)
    energy     = np.linspace(-pot_height_har, result['edges'][-1], ngirde)
    functional = kp_functional(energy, pot_height_har, width_well_bohr, width_barrier_bohr)
    fig        = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=energy * eff_har2ev,
            y=functional,
            mode='lines',
            name='Functional',
            line=dict(color='black'),
        )
    )
    fig.add_hline(y=1, line=dict(color='gray', dash='dot'))
    fig.add_hline(y=-1, line=dict(color='gray', dash='dot'))
    for j in range(num_bands):
        fig.add_vrect(x0=result['edges'][2 * j] * eff_har2ev, x1=result['edges'][2 * j + 1] * eff_har2ev,
                      fillcolor=list_color[j % len(list_color)], opacity=0.1, line_width=0)
    fig.update_layout(title='Analytic Solutions (Bloch wave)', xaxis_title='Energy [eV]',
                      yaxis_title='Functional', yaxis_range=[-2, 2], width=1000, height=400)
    fig.show()
    return result


# Plot defect eigenstates and bands with the reference-potential comparison.
def plot_defect_result(result, normal, num_bands=2):
    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.6, 0.4],
        subplot_titles=('Eigenstates & Potential', 'E–k Diagram (Numerical)'),
    )
    fig.add_trace(
        go.Scatter(
            x=result['x'],
            y=normal['potential'],
            name='Reference potential',
            line=dict(color='gray', dash='dash'),
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=result['x'], y=result['potential'], name='Potential', line=dict(color='black')
        ),
        row=1, col=1,
    )
    count         = min(DEFECT_CELLS * num_bands, result['energies'].shape[1] - 1)
    plotted_count = max(DEFECT_CELLS + 1, count)  # E7 is required for the reference gap.
    displayed = [normal['potential'], result['potential'], normal['energies'][:, :plotted_count],
                 result['energies'][:, :plotted_count]]

    ### Left subplot: Eigenstates and potential
    for state in range(count):
        energy = result['energies'][0, state]
        psi    = wavefunction_for_plot(result['gamma_vectors'][:, state])
        # Keep the same display scale, labels and band colors as the ordinary mode.
        shifted = energy + 10 * psi
        displayed.append(shifted)
        label = f'Eig {state}'
        color = list_color[(state // DEFECT_CELLS) % len(list_color)]
        fig.add_trace(
            go.Scatter(
                x=result['x'],
                y=shifted,
                mode='lines',
                name=label,
                line=dict(color=color),
                customdata=np.column_stack((np.full(len(psi), energy), psi)),
                hovertemplate=label + f' (E{state+1}); Gamma'
                '<br>x=%{x:.4f} Å<br>E=%{customdata[0]:.6f} eV'
                '<br>ψ=%{customdata[1]:.6f}'
                '<br>Display: E + 10×ψ<extra></extra>',
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=[result['x'][0], result['x'][-1]],
                y=[energy, energy],
                mode='lines',
                line=dict(color='gray', dash='dot'),
                name=label + ' energy',
                showlegend=False,
                hovertemplate=label + '<br>E=%{y:.6f} eV<extra></extra>',
            ),
            row=1, col=1,
        )

    ### Right subplot: Numerical Bloch bands and reference potential
    phase = result['phases'] / np.pi
    for state in range(plotted_count):
        band  = state // DEFECT_CELLS
        color = list_color[band % len(list_color)]
        hover = (
            f'Band {band+1}; Eig {state} (E{state+1})'
            '<br>kL/π=%{x:.4f}<br>E=%{y:.6f} eV<extra></extra>'
        )
        fig.add_trace(
            go.Scatter(
                x=phase,
                y=normal['energies'][:, state],
                mode='lines',
                line=dict(color=color, dash='dot'),
                opacity=0.35,
                name=f'Band {band+1} (Reference)',
                legendgroup=f'normal_band_{band+1}',
                showlegend=state % DEFECT_CELLS == 0,
                hovertemplate='Reference H(k)<br>' + hover,
            ),
            row=1, col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=phase,
                y=result['energies'][:, state],
                mode='lines',
                line=dict(color=color),
                name=f'Band {band+1}',
                legendgroup=f'defect_band_{band+1}',
                showlegend=state % DEFECT_CELLS == 0,
                hovertemplate='Defect H(k)<br>' + hover,
            ),
            row=1, col=2,
        )
    for edge, label in ((result['ev'], 'max E6'), (result['ec'], 'min E7')):
        fig.add_hline(y=edge, line_dash='dash', line_color='gray', annotation_text=label, row=1, col=2)
    fig.update_xaxes(title_text='Position [Angstrom]', row=1, col=1)
    fig.update_xaxes(title_text='kL / π (six-cell supercell)', range=[0, 1], row=1, col=2)
    plot_range = energy_plot_range(*displayed)
    for column in (1, 2):
        fig.update_yaxes(title_text='Energy [eV]', range=plot_range, matches='y' if column == 2 else None,
                         row=1, col=column)
    fig.update_layout(width=1200, height=600, title='Eigenstates, Potential',
                      legend=dict(groupclick='togglegroup'))
    halve_bloch_panel_widths(fig, 2)
    fig.show()


###############################################################################
# Calculation Entry Points
###############################################################################


def main(pot_shape, m_eff, pot_height_eV, width_barrier_ang, width_well_ang, num_well, num_bands,
         points_per_cell=251, nk=21, pbc_mode=False):
    """Calculate finite levels or numerical Bloch bands with optional analytic KP comparison."""
    global legacy_result
    values = [m_eff, pot_height_eV, width_barrier_ang, width_well_ang]
    if (
        not np.all(np.isfinite(values))
        or m_eff <= 0
        or pot_height_eV < 0
        or width_barrier_ang < 0
        or width_well_ang <= 0
    ):
        raise ValueError('Use positive mass/well width and nonnegative depth/barrier width.')
    if (
        pot_shape not in range(5)
        or int(num_bands) != num_bands
        or num_bands < 1
        or int(num_well) != num_well
        or not 1 <= num_well <= num_well_kriong
        or int(points_per_cell) != points_per_cell
        or points_per_cell < 15
        or int(nk) != nk
        or nk < 2
    ):
        raise ValueError('Invalid shape, well count, band count, grid or Bloch-point count.')
    num_bands, num_well, points_per_cell = int(num_bands), int(num_well), int(points_per_cell)
    is_kroing      = bool(pbc_mode)
    num_well_total = 6 if is_kroing else num_well + 2 * num_well_lr

    # Preserve existing potential geometry for each shape.
    ngridx = points_per_cell * num_well_total
    box_ang = num_well_total * (
        width_barrier_ang + width_well_ang if pot_shape == 0 else width_well_ang
    )

    ### Effective mass atomic unit conversion
    global eff_bohr, eff_ang2bohr, eff_har, eff_har2ev
    eff_bohr           = bohr_radius / m_eff
    eff_ang2bohr       = 1e-10 / eff_bohr
    eff_har            = hartree_energy * m_eff
    eff_har2ev         = eff_har / ev_energy
    box_bohr           = box_ang * eff_ang2bohr
    pot_height_har     = pot_height_eV / eff_har2ev
    width_well_bohr    = width_well_ang * eff_ang2bohr
    width_barrier_bohr = width_barrier_ang * eff_ang2bohr
    dx                 = box_bohr / ngridx

    ### Calculation
    print('Solving Schrödinger equation (sparse lowest states)')
    periodic_result, analytic_result = None, None
    if is_kroing:
        periodic_result = solve_periodic_bands(ngridx, pot_shape, pot_height_har, width_barrier_bohr,
            width_well_bohr, dx, 6, num_bands, nk)
        eigval = periodic_result['energies'][0]
        eigvec = periodic_result['gamma_vectors']
        if pot_shape == 0:
            try:
                analytic_result = solve_and_plot_analytic(pot_height_har, width_barrier_bohr, width_well_bohr,
                    num_bands, periodic_result['phases'])
            except (ValueError, RuntimeError) as exc:
                print(f'Analytic KP comparison unavailable: {exc}')
        gaps = []
        for j in range(1, num_bands):
            ev    = periodic_result['energies'][:, 6 * j - 1].max() * eff_har2ev
            ec    = periodic_result['energies'][:, 6 * j].min() * eff_har2ev
            gap   = max(0., ec - ev)
            entry = dict(bands=(j, j + 1), ev=ev, ec=ec, gap=gap)
            text  = f'Gap {j}→{j+1}: numerical {gap:.6f} eV'
            if analytic_result is not None:
                analytic_gap = max(0.,
                    (analytic_result['edges'][2 * j] - analytic_result['edges'][2 * j - 1]) * eff_har2ev)
                entry.update(analytic_gap=analytic_gap, difference=gap - analytic_gap)
                text += f'; KP {analytic_gap:.6f} eV'
            print(text)
            gaps.append(entry)
        if num_bands == 1:
            print('One band displayed; no adjacent-band gap to report.')
    else:
        eigval, eigvec = solve_tise(ngridx, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr,
                                    dx, num_well_total, False, nstates=num_well * num_bands)
        gaps = []

    ### Visualization
    plot_supercell(eigval, eigvec, ngridx, box_bohr, pot_shape, pot_height_har, width_barrier_bohr,
                   width_well_bohr, num_bands, num_well_total, is_kroing, periodic_result, analytic_result)
    legacy_result = dict(
        eigenvalues=eigval,
        eigenvectors=eigvec,
        periodic=periodic_result,
        analytic=analytic_result,
        gaps=gaps,
        points_per_cell=points_per_cell,
        nk=int(nk) if is_kroing else None,
        energy_to_ev=eff_har2ev,
    )
    print(
        f'Done! {points_per_cell} grid points/cell'
        + (f'; {nk} Bloch points; six-cell supercell.' if is_kroing else '.')
    )
    return legacy_result


# Calculate defect bands, compare the reference potential, and display the results.
def main_defect(host, specs=(), separation=1, points_per_cell=DEFECT_POINTS_PER_CELL, nk=21, num_bands=2):
    """Run a local-defect calculation from explicit settings without reading widgets."""
    global defect_result
    validate_defect_inputs(host, specs, separation, points_per_cell)
    normal               = normal_defect_reference(host, points_per_cell, nk, num_bands)
    result               = solve_defect_bands(host, specs, separation, points_per_cell, nk, num_bands)
    result['normal_gap'] = normal['gap']
    result['delta_gap']  = result['gap'] - normal['gap']
    # Count configured defect sites per periodically repeated supercell.
    # A replacement identical to the reference potential still counts as a configured site.
    length_ang = DEFECT_CELLS * (host[1] + host[2])
    result['defect_density_ang_inv'] = len(specs) / length_ang
    result['defect_density_cm_inv']  = result['defect_density_ang_inv'] * 1e8
    defect_result = result
    print(
        f'Eg_normal = {normal["gap"]:.6f} eV; Eg_defect = {result["gap"]:.6f} eV; '
        f'ΔEg = {result["delta_gap"]:+.6f} eV'
    )
    print(
        f'1D defect density = {result["defect_density_ang_inv"]:.6g} Å⁻¹ '
        f'= {result["defect_density_cm_inv"]:.6g} cm⁻¹ '
        f'({len(specs)} / {length_ang:g} Å)'
    )
    if result['raw_gap'] <= 1e-8:
        print('Gap closed / band overlap at the reference filling.')
    for index, spec in enumerate(specs, 1):
        print(
            f'Defect {index}: depth {spec[0]:g} eV '
            f'(reference depth difference {spec[0] - host[0]:+g} eV), width {spec[1]:g} Å, {spec[2]}.'
        )
    print(f'{result["nk"]} Bloch points; {result["points_per_cell"]} points/cell.')
    plot_defect_result(result, normal, num_bands)
    return defect_result


if __name__ == "__main__":
    readme = r"""
Potential V(x)

   V0 ┌───┐     ┌───┐     ┌───┐
      │   │     │   │     │   │
0─────┘   └─────┘   └─────┘   └───────► x
  ... | b |  a  | b |  a  | ...
          <─────────>
             period
    
    Barrier Height  = V₀ 
    Barrier Width   = b
    Well Width      = a
    Period of cell  = a + b 

※ Enable [PBC mode] for the periodic calculation,
    Simulator will sweep Bloch phases in a six-cell Kronig-Penney supercell.
    """
    print("-" * 50, readme, "\n", "-" * 50)

    pass


###############################################################################
# Simulation Controls
###############################################################################

kp_controls = dict(
    num_bands=IntSlider(min=1, max=8, value=2, description='# of Bands:'),
    pot_shape=Dropdown(
        options=[('Square Well', 0), ('Triangular Well', 1), ('Parabolic Well', 2), ('Coulombic well', 3),
                 ('Custom potential', 4)],
        value=0,
        description='Potential Shape: ',
    ),
    pot_height_eV=FloatSlider(min=0, max=100, step=0.1, value=3, description='Barrier Height [eV]:'),
    width_barrier_ang=FloatSlider(min=0, max=1000, step=0.1, value=3, description='Barrier Width [Ang]:'),
    width_well_ang=FloatSlider(min=0, max=1000, step=0.5, value=3, description='Well Width [Ang]:'),
    num_well=IntSlider(min=1, max=num_well_kriong, value=3, description='# of potential well:'),
    m_eff=Dropdown(
        options=[('GaAs : 0.07', 0.07), ('Silicon : 1.08', 1.08), ('Germanium : 0.55', 0.55),
                 ('Bare mass : 1.00', 1.)],
        value=1.,
        description='Effective mass',
    ),
)
for control in kp_controls.values():
    control.layout.width            = '440px'
    control.style.description_width = '175px'


# Create the depth, width, and shape controls for one defect.
def defect_widget_group(number):
    group = dict(depth=widgets.FloatText(value=3., description='Depth [eV]:'),
                 width=widgets.FloatText(value=3., description='Width [Å]:'),
                 shape=Dropdown(options=DEFECT_SHAPES, value='Square', description='Shape:'),
                 flat=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='Flat-bottom fraction:'))
    for control in group.values():
        control.layout.width            = '440px'
        control.style.description_width = '175px'
    group['box'] = widgets.VBox(
        [widgets.HTML(f'<b>Defect {number}</b>'), group['depth'], group['width'], group['shape'],
         group['flat']]
    )
    return group


# Align mode buttons with the input column of the 440 px parameter rows.
kp_pbc = widgets.ToggleButton(value=False, description='PBC (off)',
    layout=widgets.Layout(width='122px', height='30px', margin='0 0 0 8px', flex='0 0 auto'))
kp_defect_mode = widgets.ToggleButton(value=False, description='Defect (off)',
    layout=widgets.Layout(width='122px', height='30px', margin='0 0 0 8px', flex='0 0 auto'))
kp_mode_label = widgets.Label('Mode:',
                              layout=widgets.Layout(width='175px', margin='0 0 0 5px', flex='0 0 auto'))
kp_mode_row = widgets.HBox([kp_mode_label, kp_pbc, kp_defect_mode],
                           layout=widgets.Layout(width='440px', align_items='center', margin='0 0 6px 0'))
kp_defects      = [defect_widget_group(1), defect_widget_group(2)]
kp_legacy_grid  = Dropdown(options=[251, 501, 1001, 2001], value=251, description='Grid points/cell:')
kp_legacy_bloch = Dropdown(options=[21, 41, 81, 161], value=21, description='Bloch points:')
for control in (kp_legacy_grid, kp_legacy_bloch):
    control.layout.width            = '440px'
    control.style.description_width = '175px'
kp_legacy_box   = widgets.VBox([kp_legacy_grid, kp_legacy_bloch])
kp_defect_count = Dropdown(options=[1, 2], value=1, description='Defect count:')
kp_separation = Dropdown(options=[('1 cell (adjacent)', 1), ('2 cells', 2), ('3 cells (opposite)', 3)],
                         value=1, description='Separation:')
kp_independent = widgets.Checkbox(value=False, description='Independent settings for two defects')
kp_grid_points = Dropdown(options=[251, 501, 1001, 2001], value=DEFECT_POINTS_PER_CELL,
                          description='Grid points/cell:')
kp_bloch_points = Dropdown(options=[21, 41, 81, 161], value=21, description='Bloch points:')
kp_run          = widgets.Button(description='Run calculation', button_style='primary')
# Keep defect controls on the same label/input grid as the reference potential controls.
for control in (kp_defect_count, kp_separation, kp_grid_points, kp_bloch_points):
    control.layout.width            = '440px'
    control.style.description_width = '175px'
kp_independent.layout.width = '440px'
kp_run.layout               = widgets.Layout(width='244px', height='32px', margin='8px 0 8px 188px')
kp_result_output            = widgets.Output()
kp_defect_box = widgets.VBox(
    [kp_grid_points, kp_bloch_points, kp_defect_count, kp_separation, kp_independent, kp_defects[0]['box'],
     kp_defects[1]['box']]
)


# Update mode labels, visible controls, and defect-dependent settings.
def update_defect_widgets(change=None):
    # Defects belong to the periodic model; switching PBC off resets that mode.
    if not kp_pbc.value and kp_defect_mode.value:
        kp_defect_mode.value = False
    kp_pbc.description         = f'PBC ({"on" if kp_pbc.value else "off"})'
    kp_defect_mode.description = f'Defect ({"on" if kp_defect_mode.value else "off"})'
    active      = kp_pbc.value and kp_defect_mode.value
    two         = kp_defect_count.value == 2
    independent = two and kp_independent.value
    kp_defect_mode.layout.display = '' if kp_pbc.value else 'none'
    if active:
        kp_controls['pot_shape'].value = 0
    kp_controls['pot_shape'].disabled      = active
    kp_defect_box.layout.display           = '' if active else 'none'
    kp_legacy_box.layout.display           = 'none' if active else ''
    kp_legacy_bloch.layout.display         = '' if kp_pbc.value else 'none'
    kp_controls['num_well'].layout.display = 'none' if kp_pbc.value else ''
    kp_separation.layout.display           = '' if two else 'none'
    kp_independent.layout.display          = '' if two else 'none'
    kp_defects[1]['box'].layout.display    = '' if independent else 'none'
    for group in kp_defects:
        group['flat'].layout.display = '' if group['shape'].value == 'Trapezoidal' else 'none'


# Read the reference potential and replacement-well settings from the widgets.
def current_defect_settings():
    host = (kp_controls['pot_height_eV'].value, kp_controls['width_well_ang'].value,
            kp_controls['width_barrier_ang'].value, kp_controls['m_eff'].value)
    groups = [kp_defects[0]]
    if kp_defect_count.value == 2:
        groups.append(kp_defects[1] if kp_independent.value else kp_defects[0])
    specs = tuple(
        (g['depth'].value, g['width'].value, g['shape'].value, g['flat'].value) for g in groups
    )
    return host, specs, kp_separation.value


# Read the controls and call the ordinary or defect calculation entry point.
def run_kp_calculation(button=None):
    kp_run.disabled = True
    try:
        with kp_result_output:
            kp_result_output.clear_output(wait=True)
            if not (kp_pbc.value and kp_defect_mode.value):
                main(**{name: control.value for name, control in kp_controls.items()},
                     points_per_cell=kp_legacy_grid.value, nk=kp_legacy_bloch.value, pbc_mode=kp_pbc.value)
                return
            host, specs, separation = current_defect_settings()
            bands                   = kp_controls['num_bands'].value
            main_defect(host, specs, separation, kp_grid_points.value, kp_bloch_points.value, bands)
    except Exception as exc:
        with kp_result_output:
            print(f'Calculation stopped: {type(exc).__name__}: {exc}')
    finally:
        kp_run.disabled = False


for control in (kp_pbc, kp_defect_mode, kp_controls['pot_shape'], kp_controls['pot_height_eV'],
                kp_controls['width_well_ang'], kp_controls['width_barrier_ang'], kp_controls['num_well'],
                kp_defect_count, kp_separation, kp_independent):
    control.observe(update_defect_widgets, names='value')
for group in kp_defects:
    for name in ('depth', 'shape'):
        group[name].observe(update_defect_widgets, names='value')
kp_run.on_click(run_kp_calculation)
update_defect_widgets()
display(
    widgets.VBox(
        [kp_mode_row]
        + list(kp_controls.values())
        + [kp_legacy_box, kp_defect_box, kp_run, kp_result_output]
    )
)
